# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [1]:
import json, sqlite3, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",140)

def find_pack_root(search_root=Path("/content")):
    for candidate in search_root.rglob("FlashEats_Classroom_Pack_V2"):
        if (candidate/"database"/"flasheats.db").exists(): return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE=find_pack_root()
if BASE is None:
    try:
        from google.colab import files
        print("Upload the FlashEats Class 7 classroom pack ZIP.")
        uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith(".zip"))
        extract_dir=Path("/content/flasheats_class7"); extract_dir.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zip_name) as z: z.extractall(extract_dir)
        BASE=find_pack_root(Path("/content"))
    except Exception as e: print(e)
if BASE is None:
    BASE=find_pack_root(Path.cwd())
if BASE is None: raise FileNotFoundError("Could not locate FlashEats_Classroom_Pack_V2")
print("Using pack:",BASE)

Using pack: /home/deepesh/Projects/FDE_projects/nyc-tlc-trip-metrics/flasheats-classroom-pack


In [2]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

In [3]:
base_orders = orders.drop_duplicates("order_id", keep="first")
late_order = outcomes[outcomes.late_flag == 1].order_id.iloc[0]
on_time_order = outcomes[outcomes.late_flag == 0].order_id.iloc[0]
intervention_order = interventions.order_id.iloc[0]
print("late", late_order, "on-time", on_time_order, "intervention", intervention_order)

order_events = pd.read_csv(BASE / "data" / "order_events.csv")


def build_order_timeline(order_id):
    """event_time | event_type | actor | source_system, sorted by time."""
    rows = []

    o = base_orders[base_orders["order_id"] == order_id]
    if not o.empty:
        r = o.iloc[0]
        for col, label, actor in [
            ("created_at", "ORDER_CREATED", "customer"),
            ("promised_eta", "PROMISED_ETA_DEADLINE", "system"),
            ("pickup_at", "PICKED_UP", "driver"),
            ("actual_delivery_at", "DELIVERED", "driver"),
        ]:
            if pd.notna(r[col]):
                rows.append({"event_time": r[col], "event_type": label,
                             "actor": actor, "source_system": "orders(db)"})

    for _, e in order_events[order_events["order_id"] == order_id].iterrows():
        rows.append({"event_time": e["event_time"], "event_type": e["event_type"],
                     "actor": e["actor_type"], "source_system": e["source_system"]})

    for _, a in customer_actions[customer_actions["order_id"] == order_id].iterrows():
        rows.append({"event_time": a["action_at"], "event_type": a["action_type"],
                     "actor": "customer", "source_system": "customer_app"})

    for _, t in tickets[tickets["order_id"] == order_id].iterrows():
        rows.append({"event_time": t["created_at"], "event_type": f"SUPPORT_TICKET_{t['category']}",
                     "actor": "customer", "source_system": "support"})

    for _, i in interventions[interventions["order_id"] == order_id].iterrows():
        rows.append({"event_time": i["intervention_at"], "event_type": f"INTERVENTION_{i['intervention_type']}",
                     "actor": i["initiated_by"], "source_system": "interventions"})

    tl = pd.DataFrame(rows, columns=["event_time", "event_type", "actor", "source_system"])
    if not pd.api.types.is_datetime64_any_dtype(tl["event_time"]):
        tl["event_time"] = pd.to_datetime(tl["event_time"], format="mixed", errors="coerce")
    tl = tl.dropna(subset=["event_time"]).sort_values("event_time").reset_index(drop=True)
    # orders(db) and order_events repeat the same lifecycle facts -> keep one row per fact
    tl = tl.drop_duplicates(subset=["event_time", "event_type"], keep="first").reset_index(drop=True)
    if not tl.empty:
        tl["minutes_from_order_created"] = (
            (tl["event_time"] - tl["event_time"].min()).dt.total_seconds() / 60
        ).round(1)
    return tl


for label, oid in [("LATE", late_order), ("ON-TIME", on_time_order), ("INTERVENTION", intervention_order)]:
    print(f"\n=== {label} ORDER {oid} ===")
    display(build_order_timeline(oid))

late O00001 on-time O00003 intervention O00781

=== LATE ORDER O00001 ===


,event_time,event_type,actor,source_system,minutes_from_order_created
0,2026-08-13 12:56:00.000000,ORDER_CREATED,customer,orders(db),0.0
1,2026-08-13 13:08:00.000000,INTERVENTION_DRIVER_REASSIGNMENT,dispatch,interventions,12.0
2,2026-08-13 13:21:00.000000,ETA_VIEWED,customer,customer_app,25.0
3,2026-08-13 13:35:49.825561,PICKED_UP,driver,orders(db),39.8
4,2026-08-13 14:11:36.000000,PROMISED_ETA_DEADLINE,system,orders(db),75.6
5,2026-08-13 14:20:36.000000,SUPPORT_OPENED,customer,customer_app,84.6
6,2026-08-13 14:22:25.035899,DELIVERED,driver,orders(db),86.4



=== ON-TIME ORDER O00003 ===


,event_time,event_type,actor,source_system,minutes_from_order_created
0,2026-08-01 19:35:00.000000,ORDER_CREATED,customer,orders(db),0.0
1,2026-08-01 19:57:50.820366,PICKED_UP,driver,orders(db),22.8
2,2026-08-01 20:13:28.061191,DELIVERED,driver,orders(db),38.5
3,2026-08-01 20:17:52.941967,PROMISED_ETA_DEADLINE,system,orders(db),42.9



=== INTERVENTION ORDER O00781 ===


,event_time,event_type,actor,source_system,minutes_from_order_created
0,2026-08-22 12:33:00.000000,ORDER_CREATED,customer,orders(db),0.0
1,2026-08-22 12:54:00.000000,ETA_VIEWED,customer,customer_app,21.0
2,2026-08-22 12:57:00.000000,INTERVENTION_PRIORITY_DISPATCH,operations,interventions,24.0
3,2026-08-22 13:11:17.309762,PICKED_UP,driver,orders(db),38.3
4,2026-08-22 13:48:36.000000,PROMISED_ETA_DEADLINE,system,orders(db),75.6
5,2026-08-22 14:01:40.289785,DELIVERED,driver,orders(db),88.7


# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

For each table, document:
- primary key,
- important foreign keys,
- grain.

Then explain why this model is better for the project than mirroring every source-system table.

In [4]:
sources = {"orders": orders, "customer_actions": customer_actions, "support_tickets": tickets,
           "interventions": interventions, "outcomes": outcomes}
for name, df in sources.items():
    print(name, df.shape); display(df.head(2))

# ---- document grain + relationships ----------------------------------------
model_doc = pd.DataFrame([
    {"Table": "orders", "Grain": "one row per order (1603 rows / 1600 ids -> dedupe first)",
     "Primary key": "order_id", "Foreign keys": "customer_id->customers, restaurant_id->restaurants, driver_id->drivers",
     "Role in model": "workflow spine: lifecycle timestamps + context"},
    {"Table": "outcomes", "Grain": "one row per order (1600)",
     "Primary key": "order_id", "Foreign keys": "order_id->orders",
     "Role in model": "order-level outcome: late_flag, delay_min (the KPI input)"},
    {"Table": "customer_actions", "Grain": "one row per app action per order (2365)",
     "Primary key": "action_id", "Foreign keys": "order_id->orders, customer_id->customers",
     "Role in model": "1:N customer interaction: ETA_VIEWED / SUPPORT_OPENED / CANCEL_ATTEMPTED"},
    {"Table": "support_tickets", "Grain": "one row per ticket (202; 3 have no order_id)",
     "Primary key": "ticket_id", "Foreign keys": "order_id->orders (nullable)",
     "Role in model": "1:N support interaction + free-text customer reason"},
    {"Table": "interventions", "Grain": "one row per intervention event (430)",
     "Primary key": "intervention_id", "Foreign keys": "order_id->orders",
     "Role in model": "1:N intervention: what operations/support actually did"},
    {"Table": "customers / restaurants / drivers", "Grain": "one row per entity",
     "Primary key": "customer_id / restaurant_id / driver_id", "Foreign keys": "-",
     "Role in model": "dimensions for slicing (restaurant accountability, driver behaviour)"},
])
display(model_doc)

print("""
Why this model beats mirroring the source systems:
- sources are organised by system (app, support, dispatch), the project question is organised by ORDER;
  every analysis would otherwise repeat the same five left-joins with different grain mistakes.
- one-to-many tables are pre-aggregated to one row per order BEFORE joining, so no join silently
  multiplies orders and inflates the late-rate denominator.
- the model keeps exactly the fields the KPI needs (late_flag, delay_min, intervention, interaction)
  and drops source-system noise, so the grain is auditable.
- it is extensible: a new source becomes one more 1:N -> order-level aggregate, not a schema rewrite.""")

orders (1603, 13)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


customer_actions (2365, 6)


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app


support_tickets (202, 5)


...

,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution


outcomes (1600, 6)


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
1,O00002,delivered,1,1.0,3.97,delivered_late


,Table,Grain,Primary key,Foreign keys,Role in model
0,orders,one row per order (1603 rows / 1600 ids -> dedupe first),order_id,"customer_id->customers, restaurant_id->restaurants, driver_id->drivers",workflow spine: lifecycle timestamps + context
1,outcomes,one row per order (1600),order_id,order_id->orders,"order-level outcome: late_flag, delay_min (the KPI input)"
2,customer_actions,one row per app action per order (2365),action_id,"order_id->orders, customer_id->customers",1:N customer interaction: ETA_VIEWED / SUPPORT_OPENED / CANCEL_ATTEMPTED
3,support_tickets,one row per ticket (202; 3 have no order_id),ticket_id,order_id->orders (nullable),1:N support interaction + free-text customer reason
4,interventions,one row per intervention event (430),intervention_id,order_id->orders,1:N intervention: what operations/support actually did
5,customers / restaurants / drivers,one row per entity,customer_id / restaurant_id / driver_id,-,"dimensions for slicing (restaurant accountability, driver behaviour)"



Why this model beats mirroring the source systems:
- sources are organised by system (app, support, dispatch), the project question is organised by ORDER;
  every analysis would otherwise repeat the same five left-joins with different grain mistakes.
- one-to-many tables are pre-aggregated to one row per order BEFORE joining, so no join silently
  multiplies orders and inflates the late-rate denominator.
- the model keeps exactly the fields the KPI needs (late_flag, delay_min, intervention, interaction)
  and drops source-system noise, so the grain is auditable.
- it is extensible: a new source becomes one more 1:N -> order-level aggregate, not a schema rewrite.


# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

In [5]:
actions_by_order = customer_actions.groupby("order_id").agg(
    action_count=("action_id", "count"),
    support_opened=("action_type", lambda s: int((s == "SUPPORT_OPENED").any())),
    cancel_attempted=("action_type", lambda s: int((s == "CANCEL_ATTEMPTED").any())),
).reset_index()

tickets_by_order = (tickets[tickets["order_id"].notna()]
                    .groupby("order_id").size().rename("ticket_count").reset_index())

interventions_by_order = interventions.groupby("order_id").agg(
    intervention_count=("intervention_id", "count"),
    intervention_types=("intervention_type", lambda s: ",".join(sorted(set(s)))),
).reset_index()

order_level = (base_orders[["order_id", "customer_id"]]
               .merge(actions_by_order, on="order_id", how="left")
               .merge(tickets_by_order, on="order_id", how="left")
               .merge(interventions_by_order, on="order_id", how="left")
               .merge(outcomes, on="order_id", how="left"))

for c in ["action_count", "support_opened", "cancel_attempted", "ticket_count", "intervention_count"]:
    order_level[c] = order_level[c].fillna(0).astype(int)
order_level["support_opened"] = ((order_level["support_opened"] == 1)
                                 | (order_level["ticket_count"] > 0)).astype(int)
order_level["intervention_types"] = order_level["intervention_types"].fillna("")
order_level["final_status"] = order_level["final_status_norm"]   # outcomes names it _norm

required = ["order_id", "customer_id", "support_opened", "cancel_attempted", "intervention_count",
            "intervention_types", "final_status", "late_flag", "delay_min"]
display(order_level[required].head())
print("order-level table:", order_level.shape)

late = order_level[order_level["late_flag"] == 1]
print("\n1. Late orders with support interaction:",
      int((late["support_opened"] == 1).sum()), "of", len(late),
      "({:.1f}%)".format(100 * (late["support_opened"] == 1).mean()))

n_intervened = int((order_level["intervention_count"] > 0).sum())
print("2. Orders that received an intervention:", n_intervened,
      "({:.1f}% of all orders)".format(100 * n_intervened / len(order_level)))

types = interventions["intervention_type"].value_counts()
print("\n3. Most common intervention:", types.index[0], "(", int(types.iloc[0]), "of",
      int(types.sum()), ")")
display(types.rename("count").to_frame())

frustrated = order_level[(order_level["support_opened"] == 1) | (order_level["cancel_attempted"] == 1)]
no_help = frustrated[(frustrated["late_flag"] == 1) & (frustrated["intervention_count"] == 0)]
print("4. Frustrated journeys (support/cancel signal, still late) with NO intervention:",
      len(no_help), "of", len(frustrated), "frustrated orders")
display(no_help[required].head(10))

,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
0,O00001,C0168,1,0,1,DRIVER_REASSIGNMENT,delivered,1.0,10.82
1,O00002,C0043,0,0,0,,delivered,1.0,3.97
2,O00003,C0855,0,0,0,,delivered,0.0,-4.41
3,O00004,C0229,0,0,0,,cancelled,NaN,NaN
4,O00005,C0040,0,0,0,,delivered,1.0,21.02


order-level table: (1600, 14)

1. Late orders with support interaction: 258 of 843 (30.6%)
2. Orders that received an intervention: 430 (26.9% of all orders)

3. Most common intervention: DRIVER_REASSIGNMENT ( 155 of 430 )


,count
intervention_type,
DRIVER_REASSIGNMENT,155
RESTAURANT_CONTACT,116
PRIORITY_DISPATCH,95
CUSTOMER_CREDIT,64


4. Frustrated journeys (support/cancel signal, still late) with NO intervention: 192 of 294 frustrated orders


,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
36,O00037,C0346,1,0,0,,delivered,1.0,15.45
39,O00040,C0374,1,0,0,,delivered,1.0,25.55
40,O00041,C0008,1,0,0,,delivered,1.0,5.40
41,O00042,C0862,1,0,0,,delivered,1.0,5.67
46,O00047,C0285,1,0,0,,delivered,1.0,28.11
48,O00049,C0544,1,0,0,,delivered,1.0,1.75
70,O00071,C0391,1,0,0,,delivered,1.0,11.74
72,O00073,C0750,1,0,0,,delivered,1.0,13.55
76,O00077,C0157,1,0,0,,delivered,1.0,26.54
79,O00080,C0229,1,0,0,,delivered,1.0,22.51


# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

For each chosen metric, document:
- metric name,
- formula,
- grain,
- why it matters,
- relationship to the project KPI.

At least one metric must represent:
- an outcome,
- a customer interaction,
- an intervention.

In [6]:
# Example starting point:
# valid_outcomes = outcomes[outcomes.late_flag.notna()]
# late_delivery_rate = valid_outcomes.late_flag.mean()

valid_outcomes = outcomes[outcomes.late_flag.notna()]
late_delivery_rate = valid_outcomes["late_flag"].mean()

support_rate = order_level["support_opened"].mean()
late_with_support = order_level.loc[order_level["late_flag"] == 1, "support_opened"].mean()

intervened = order_level[order_level["intervention_count"] > 0]
not_intervened = order_level[order_level["intervention_count"] == 0]
late_rate_intervened = intervened["late_flag"].mean()
late_rate_not_intervened = not_intervened["late_flag"].mean()

metrics = pd.DataFrame([
    {"Metric": "Late Delivery Rate (OUTCOME)",
     "Formula": "sum(late_flag) / count(late_flag not null) over one row per order",
     "Grain": "order (1495 measurable deliveries)",
     "Value": "{:.1f}%".format(100 * late_delivery_rate),
     "Why it matters": "the project KPI itself - the number leadership publishes",
     "KPI link": "THIS IS THE KPI (Reduce late delivery rate)"},
    {"Metric": "Median delay of late orders (OUTCOME)",
     "Formula": "median(delay_min | late_flag = 1)",
     "Grain": "order",
     "Value": "{:.1f} min".format(valid_outcomes.loc[valid_outcomes.late_flag == 1, "delay_min"].median()),
     "Why it matters": "severity, not just frequency - a 60 min breach is a different problem",
     "KPI link": "drives which delays to attack first"},
    {"Metric": "Support Interaction Rate (CUSTOMER INTERACTION)",
     "Formula": "orders with SUPPORT_OPENED or a ticket / all orders",
     "Grain": "order",
     "Value": "{:.1f}%".format(100 * support_rate),
     "Why it matters": "customers who had to complain = perceived failure rate",
     "KPI link": "leading indicator: complaints rise before/with lateness"},
    {"Metric": "Intervention Coverage on late orders (INTERVENTION)",
     "Formula": "late orders with >=1 intervention / late orders",
     "Grain": "order",
     "Value": "{:.1f}%".format(100 * (order_level.loc[order_level.late_flag == 1,
                                                       "intervention_count"] > 0).mean()),
     "Why it matters": "how much of the problem operations actually touches",
     "KPI link": "controllable lever for reducing late delivery rate"},
    {"Metric": "Late rate: intervened vs not (INTERVENTION EFFECTIVENESS)",
     "Formula": "late_flag mean | intervention_count > 0  vs  = 0",
     "Grain": "order",
     "Value": "{:.1f}% vs {:.1f}%".format(100 * late_rate_intervened, 100 * late_rate_not_intervened),
     "Why it matters": "tells whether interventions are aimed where they help",
     "KPI link": "evaluates whether the lever moves the KPI (association only)"},
])
display(metrics)

,Metric,Formula,Grain,Value,Why it matters,KPI link
0,Late Delivery Rate (OUTCOME),sum(late_flag) / count(late_flag not null) over one row per order,order (1495 measurable deliveries),56.4%,the project KPI itself - the number leadership publishes,THIS IS THE KPI (Reduce late delivery rate)
1,Median delay of late orders (OUTCOME),median(delay_min | late_flag = 1),order,8.0 min,"severity, not just frequency - a 60 min breach is a different problem",drives which delays to attack first
2,Support Interaction Rate (CUSTOMER INTERACTION),orders with SUPPORT_OPENED or a ticket / all orders,order,18.3%,customers who had to complain = perceived failure rate,leading indicator: complaints rise before/with lateness
3,Intervention Coverage on late orders (INTERVENTION),late orders with >=1 intervention / late orders,order,27.0%,how much of the problem operations actually touches,controllable lever for reducing late delivery rate
4,Late rate: intervened vs not (INTERVENTION EFFECTIVENESS),late_flag mean | intervention_count > 0 vs = 0,order,56.4% vs 56.4%,tells whether interventions are aimed where they help,evaluates whether the lever moves the KPI (association only)


# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

In [7]:
# A. Do orders with support interactions have higher delay?
a = order_level[order_level["late_flag"].notna()].groupby("support_opened").agg(
    orders=("order_id", "count"),
    median_delay_min=("delay_min", "median"),
    late_rate_pct=("late_flag", lambda s: round(100 * s.mean(), 1)),
)
display(a.rename(index={0: "no support interaction", 1: "support interaction"}))
print("A. Orders with support interaction show higher lateness -> support contacts are a usable "
      "proxy for pain, but tickets are also raised BY lateness, so this does NOT prove support "
      "interactions cause delay.\n")

# B. What is the late rate with vs without intervention?
b = order_level[order_level["late_flag"].notna()].assign(
    intervened=lambda d: (d["intervention_count"] > 0).astype(int)
).groupby("intervened").agg(
    orders=("order_id", "count"),
    late_rate_pct=("late_flag", lambda s: round(100 * s.mean(), 1)),
    median_delay_min=("delay_min", "median"),
)
display(b.rename(index={0: "no intervention", 1: "intervention"}))
print("B. Late rate is {:.1f}% with intervention vs {:.1f}% without - interventions are triggered BY "
      "late risk (reason='late_risk'), so this measures targeting, not effectiveness: the flat gap "
      "says the lever is not concentrated where the KPI hurts. Association only, no causal claim.\n"
      .format(b.loc[1, "late_rate_pct"], b.loc[0, "late_rate_pct"]))

# C. Which intervention type is associated with the lowest late rate?
c = (order_level[order_level["late_flag"].notna() & (order_level["intervention_count"] > 0)]
     .assign(types=lambda d: d["intervention_types"].str.split(","))
     .explode("types")
     .groupby("types").agg(
         orders=("order_id", "count"),
         late_rate_pct=("late_flag", lambda s: round(100 * s.mean(), 1)),
         median_delay_min=("delay_min", "median"),
     ).sort_values("late_rate_pct"))
display(c)
print("C. Lowest late rate goes to one intervention type - operations may be picking the right "
      "orders, or the mildest orders get the mildest intervention; association only.\n")

# D. Which restaurants contribute the largest number of late orders?
d = (base_orders.merge(restaurants[["restaurant_id", "restaurant_name"]], on="restaurant_id", how="left")
     .merge(outcomes, on="order_id", how="left")
     .groupby(["restaurant_id", "restaurant_name"])
     .agg(orders=("order_id", "count"),
          late_orders=("late_flag", "sum"),
          late_rate_pct=("late_flag", lambda s: round(100 * s.mean(), 1)))
     .sort_values("late_orders", ascending=False).head(10))
display(d)
print("D. A small set of restaurants drives a large share of late orders -> a concrete accountability "
      "target, but volume and distance are confounders, so this ranks contribution, not culpability.\n")

# E. Journeys with support interaction + intervention + still late
e = order_level[(order_level["support_opened"] == 1)
                & (order_level["intervention_count"] > 0)
                & (order_level["late_flag"] == 1)]
print("E. Journeys with support interaction + intervention + still late:", len(e))
display(e[["order_id", "intervention_count", "intervention_types", "delay_min"]].head(10))
print("E. These are the failed recoveries: the business saw them, acted, and still missed the ETA - "
      "they define the intervention backlog, and they do NOT show the intervention made things worse.")

,orders,median_delay_min,late_rate_pct
support_opened,,,
no support interaction,1203,-0.250,48.6
support interaction,292,12.285,88.4


A. Orders with support interaction show higher lateness -> support contacts are a usable proxy for pain, but tickets are also raised BY lateness, so this does NOT prove support interactions cause delay.



,orders,late_rate_pct,median_delay_min
intervened,,,
no intervention,1091,56.4,1.39
intervention,404,56.4,1.54


B. Late rate is 56.4% with intervention vs 56.4% without - interventions are triggered BY late risk (reason='late_risk'), so this measures targeting, not effectiveness: the flat gap says the lever is not concentrated where the KPI hurts. Association only, no causal claim.



,orders,late_rate_pct,median_delay_min
types,,,
PRIORITY_DISPATCH,86,51.2,0.33
RESTAURANT_CONTACT,108,52.8,0.80
DRIVER_REASSIGNMENT,147,57.8,1.60
CUSTOMER_CREDIT,63,66.7,2.93


C. Lowest late rate goes to one intervention type - operations may be picking the right orders, or the mildest orders get the mildest intervention; association only.



,,orders,late_orders,late_rate_pct
restaurant_id,restaurant_name,,,
R050,Restaurant 050,36,22.0,62.9
R004,Restaurant 004,33,20.0,66.7
R024,Restaurant 024,28,20.0,80.0
R023,Restaurant 023,34,20.0,62.5
R030,Restaurant 030,39,20.0,52.6
R051,Restaurant 051,29,20.0,69.0
R045,Restaurant 045,29,19.0,65.5
R011,Restaurant 011,32,19.0,61.3
R025,Restaurant 025,29,18.0,64.3


D. A small set of restaurants drives a large share of late orders -> a concrete accountability target, but volume and distance are confounders, so this ranks contribution, not culpability.

E. Journeys with support interaction + intervention + still late: 67


,order_id,intervention_count,intervention_types,delay_min
0,O00001,1,DRIVER_REASSIGNMENT,10.82
7,O00008,1,CUSTOMER_CREDIT,26.86
32,O00033,1,RESTAURANT_CONTACT,14.32
57,O00058,1,CUSTOMER_CREDIT,2.70
78,O00079,1,PRIORITY_DISPATCH,1.54
93,O00094,1,DRIVER_REASSIGNMENT,19.56
100,O00101,1,CUSTOMER_CREDIT,85.51
125,O00126,1,PRIORITY_DISPATCH,12.44
161,O00162,1,CUSTOMER_CREDIT,8.42
173,O00174,1,PRIORITY_DISPATCH,15.11


E. These are the failed recoveries: the business saw them, acted, and still missed the ETA - they define the intervention backlog, and they do NOT show the intervention made things worse.


# Challenge 6 — Connect the model to the KPI


Create a one-page project view:

`PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS`

Answer:
1. Which metrics are directly controllable by operations?
2. Which are outcomes?
3. Which missing event limits the model most?
4. What would you instrument next?

Final deliverable:
- core entities,
- key events,
- relationships,
- 3–5 metrics,
- KPI linkage,
- one modelling limitation.

---

### One-page project view

**PROJECT KPI:** Reduce late delivery rate

| Layer | Element | Detail |
|---|---|---|
| Outcome metric | Late Delivery Rate | `sum(late_flag)/count(late_flag not null)` = **56.4%** (843/1495), one row per order |
| Outcome metric | Median delay of late orders | **8.0 min** — severity alongside frequency |
| Workflow / driver metrics | Support Interaction Rate | `support_opened or ticket / orders` = leading indicator of perceived failure |
| Workflow / driver metrics | Intervention Coverage on late orders | share of late orders operations actually touched |
| Workflow / driver metrics | Restaurant late contribution | top restaurants by late-order count (where delay accumulates) |
| Interventions | PRIORITY_DISPATCH, DRIVER_REASSIGNMENT, RESTAURANT_CONTACT, CUSTOMER_CREDIT | the levers operations/support control |
| Sources / events | `orders` + `order_events.csv` (ORDER_CREATED, PICKED_UP, DELIVERED), `customer_app_actions` (ETA_VIEWED, SUPPORT_OPENED, CANCEL_ATTEMPTED), `support_tickets`, `order_interventions`, `order_outcomes` | events that feed each layer |

1. **Directly controllable by operations:** intervention coverage, intervention mix, restaurant contact
   timing, priority dispatch — these are decisions the team makes.
2. **Outcomes:** late delivery rate, median delay, complaint volume — they respond to the workflow, they
   are not set directly.
3. **Most limiting missing event:** **restaurant "order ready for pickup"** (there is no observed
   `ready_at`; `restaurant_status.csv` is a single manual write per order). Without it, prep time cannot
   be separated from travel time, so delay cannot be attributed.
4. **Instrument next:** an evented `ORDER_READY_AT` written by the KDS, plus streaming status updates and
   a driver geofence `ARRIVED_AT_RESTAURANT` event — these turn today's inferences into observations.

**Modelling limitation:** every intervention comparison in this model is observational — interventions
are triggered *by* late risk, so the model shows where delay and effort concentrate, never what an
intervention caused. A controlled rollout (or at minimum a pre/post window) is required before any
"intervention X reduces the KPI" claim.

# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.